In [0]:
%pip install lightgbm shap scikit-learn==1.6.1 cloudpickle==3.1.1

# Deploy do Modelo LightGBM (pyfunc)

Deploy do modelo LightGBM para serving real-time via REST.

**Workflow:**
1. Busca o melhor run LightGBM no MLflow (maior PR-AUC)
2. Carrega o modelo + recria o preprocessor (ColumnTransformer)
3. Valida no conjunto de teste
4. Registra no Unity Catalog via `mlflow.pyfunc.log_model()`
5. O endpoint recebe as 152 features diretamente no payload

**UC:** `fraud_detection_dev.models.lgbm_fraud_model`

## Resumo

* Modelo: `fraud_detection_dev.models.lgbm_fraud_model` @champion
* Tipo: pyfunc (PythonModel) — features direto no payload
* Algoritmo: LightGBM + sklearn ColumnTransformer
* Threshold: 0.83

**Serving (REST):**
```python
from databricks.sdk import WorkspaceClient
import pandas as pd

w = WorkspaceClient()
sample = pd.DataFrame([{...features...}])
records = [{k: (None if pd.isna(v) else v) for k, v in r.items()} for r in sample.to_dict('records')]
response = w.api_client.do('POST', '/serving-endpoints/model_fraud/invocations',
    body={'dataframe_records': records},
    headers={'Content-Type': 'application/json'})
# response['predictions'] = [{probability_fraud, is_fraud}]
```

In [0]:
import sys, os
_nb_path = dbutils.notebook.entry_point.getNotebookPath()
for _i in range(len(_nb_path.split("/")), 0, -1):
    _r = "/".join(_nb_path.split("/")[:_i])
    if os.path.isdir(f"{_r}/src/databricks_fraud_model"):
        sys.path.insert(0, f"{_r}/src")
        break
from databricks_fraud_model.utils import load_config
import pandas as pd
import mlflow
import logging
from mlflow import MlflowClient

cfg = load_config()

FEATURE_TABLE = cfg["feature_store"]["table"]
HPO_EXPERIMENT_NAME = cfg["experiments"]["hpo"]
DEPLOY_EXPERIMENT_NAME = cfg["experiments"]["deploy"]
OPTIMAL_THRESHOLD = cfg["model"]["optimal_threshold"]
UC_MODEL_NAME = cfg["model"]["registered_name"]

logging.getLogger("mlflow.tracking.context.registry").disabled = True

experiment = mlflow.get_experiment_by_name(HPO_EXPERIMENT_NAME)
if experiment is None:
    raise ValueError(f"Experimento não encontrado: {HPO_EXPERIMENT_NAME}.")

client = MlflowClient()
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="metrics.pr_auc > 0",
    order_by=["metrics.pr_auc DESC"],
    max_results=1,
)
if not runs:
    raise ValueError(f"Nenhum run em {HPO_EXPERIMENT_NAME}. Execute o notebook 03 primeiro.")

best_run = runs[0]
best_run_id = best_run.info.run_id
print(f"Melhor Run ID: {best_run_id}")

In [0]:
df_test = spark.read.table(FEATURE_TABLE).filter("split = 'test'").toPandas()
X_test = df_test.drop(columns=["TransactionID", "target", "split"])
y_test = df_test["target"]
print(f"Teste: {len(X_test):,} linhas, {len(X_test.columns)} features, {y_test.sum():,} fraudes")

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer

parent_runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.mlflow.runName = 'LightGBM_Optuna_HPO'",
    order_by=["attributes.start_time DESC"],
    max_results=1,
)
if not parent_runs:
    raise ValueError("Run pai 'LightGBM_Optuna_HPO' não encontrado.")

model_run = parent_runs[0]
model_run_id = model_run.info.run_id
lgbm_model = mlflow.lightgbm.load_model(f"runs:/{model_run_id}/model")

exclude_cols = set(cfg["preprocessing"]["exclude_cols"]) | {"split"}
feature_names = [c for c in X_test.columns if c not in exclude_cols]
cat_cols = [c for c in feature_names if X_test[c].dtype == "object"]
num_cols = [c for c in feature_names if c not in cat_cols]

preprocessor = ColumnTransformer([
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), cat_cols),
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), num_cols),
])

df_train = spark.read.table(FEATURE_TABLE).filter("split = 'train'").toPandas()
preprocessor.fit(df_train[feature_names])
print(f"Modelo carregado: {len(feature_names)} features ({len(cat_cols)} cat, {len(num_cols)} num)")

In [0]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, confusion_matrix
)

y_proba_fraud = lgbm_model.predict_proba(preprocessor.transform(X_test[feature_names]))[:, 1]
y_pred = (y_proba_fraud >= OPTIMAL_THRESHOLD).astype(int)

test_metrics = {
    "pr_auc": average_precision_score(y_test, y_proba_fraud),
    "roc_auc": roc_auc_score(y_test, y_proba_fraud),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
}
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print(f"PR-AUC: {test_metrics['pr_auc']:.4f} | ROC-AUC: {test_metrics['roc_auc']:.4f} | F1: {test_metrics['f1_score']:.4f}")
print(f"TN={tn:,} FP={fp:,} FN={fn:,} TP={tp:,}")

In [0]:
import mlflow
import mlflow.pyfunc
import pickle, json, tempfile, os
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

UC_MODEL_NAME = cfg["model"]["registered_name"]
OPTIMAL_THRESHOLD = cfg["model"]["optimal_threshold"]

mlflow.set_experiment(cfg["experiments"]["deploy"])
mlflow.set_registry_uri("databricks-uc")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_MODEL_NAME.rsplit('.', 1)[0]}")

class FraudDetectionModel(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        with open(context.artifacts["preprocessor"], "rb") as f:
            self.preprocessor = pickle.load(f)
        with open(context.artifacts["model"], "rb") as f:
            self.model = pickle.load(f)
        with open(context.artifacts["config"], "r") as f:
            config = json.load(f)
            self.threshold = config["threshold"]
            self.feature_names = config["feature_names"]

    def predict(self, context, model_input, params=None):
        X = model_input[self.feature_names]
        proba = self.model.predict_proba(self.preprocessor.transform(X))[:, 1]
        return pd.DataFrame({
            "probability_fraud": proba,
            "is_fraud": (proba >= self.threshold).astype(int),
        })

with tempfile.TemporaryDirectory() as tmp:
    for name, obj in [("preprocessor", preprocessor), ("model", lgbm_model)]:
        with open(os.path.join(tmp, f"{name}.pkl"), "wb") as f:
            pickle.dump(obj, f)
    with open(os.path.join(tmp, "config.json"), "w") as f:
        json.dump({"threshold": OPTIMAL_THRESHOLD, "feature_names": feature_names}, f)

    sample_input = X_test[feature_names].head(100)
    sample_proba = lgbm_model.predict_proba(preprocessor.transform(sample_input))[:, 1]
    signature = infer_signature(sample_input, pd.DataFrame({
        "probability_fraud": sample_proba,
        "is_fraud": (sample_proba >= OPTIMAL_THRESHOLD).astype(int),
    }))

    # Feature importance (LightGBM nativo)
    feat_cols_pre = cat_cols + num_cols
    feat_importance = pd.DataFrame({
        "feature": feat_cols_pre,
        "importance": lgbm_model.feature_importances_,
    }).sort_values("importance", ascending=False)
    feat_importance.to_csv(os.path.join(tmp, "feature_importance.csv"), index=False)
    print(f"Feature importance salvo: top 5 = {feat_importance.head()['feature'].tolist()}")

    # SHAP values (amostra do teste)
    import shap
    X_shap = preprocessor.transform(X_test[feature_names].head(500))
    explainer = shap.TreeExplainer(lgbm_model)
    shap_values = explainer.shap_values(X_shap)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    shap_df = pd.DataFrame(shap_values, columns=feat_cols_pre)
    shap_df.to_csv(os.path.join(tmp, "shap_values.csv"), index=False)
    print(f"SHAP values salvo: {shap_df.shape[0]} amostras, {shap_df.shape[1]} features")

    with mlflow.start_run(run_name="fraud_detection_pyfunc_lgbm_v1") as run:
        mlflow.log_params(best_run.data.params)
        mlflow.log_param("optimal_threshold", OPTIMAL_THRESHOLD)
        mlflow.log_param("training_run_id", model_run_id)
        for k, v in test_metrics.items():
            mlflow.log_metric(f"test_{k}", v)

        mlflow.pyfunc.log_model(
            python_model=FraudDetectionModel(),
            artifact_path="model",
            signature=signature,
            input_example=sample_input.head(3),
            artifacts={"preprocessor": os.path.join(tmp, "preprocessor.pkl"),
                       "model": os.path.join(tmp, "model.pkl"),
                       "config": os.path.join(tmp, "config.json"),
                       "feature_importance": os.path.join(tmp, "feature_importance.csv"),
                       "shap_values": os.path.join(tmp, "shap_values.csv")},
            
            pip_requirements=[
                "mlflow==3.8.1",
                "lightgbm==4.7.0",
                "scikit-learn==1.6.1",
                "pandas==2.2.3",
                "numpy==2.1.3",
                "cloudpickle==3.1.1",
            ],
            
            registered_model_name=UC_MODEL_NAME,
        )
        deploy_run_id = run.info.run_id

client = MlflowClient(registry_uri="databricks-uc")
latest = max(client.search_model_versions(f"name='{UC_MODEL_NAME}'"), key=lambda v: int(v.version))
model_version = latest.version
for tag, val in [("algorithm", "LightGBM"), ("optimal_threshold", str(OPTIMAL_THRESHOLD)),
                 ("test_pr_auc", f"{test_metrics['pr_auc']:.4f}"),
                 ("test_f1_score", f"{test_metrics['f1_score']:.4f}")]:
    client.set_model_version_tag(UC_MODEL_NAME, model_version, tag, val)
client.set_registered_model_alias(UC_MODEL_NAME, "champion", int(model_version))
print(f"v{model_version} @champion | run {deploy_run_id[:8]}")